# Fair Voting (In-processing)

This tutorial demonstrates the fairness-aware voting strategy implemented in PipeGenie.

Unlike the standard VotingClassifier, FairVotingClassifier assigns a weight to each estimator based on both predictive performance and fairness. The final prediction is obtained through a weighted vote instead of a simple majority vote.

## Load and prepare the Dataset

This tutorial uses the employee promotion dataset.
The sensitive attribute is **education**, while the prediction target is **is_promoted**.

In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/employee_promotion.csv")

y = df["is_promoted"]

df["education"] = df["education"].fillna("Unknown")
A = df["education"]

X = df.drop(columns=["is_promoted","education"])
X = pd.get_dummies(X, drop_first=True)

## Split the Dataset

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## Create a PipeGenie Classifier with Fair Voting

Setting `fair_voting=True` replaces the standard VotingClassifier with FairVotingClassifier.

The parameter `alpha` controls the trade-off between predictive performance and fairness.

- alpha = 1 → only accuracy
- alpha = 0 → only fairness

In [ ]:
grammar_file = "../../grammars/sample-grammar-classification.xml"

from pipegenie.fairness import PipegenieClassifier

model = PipegenieClassifier(
        grammar=grammar_file,
        generations=5,
        pop_size=50,
        elite_size=5,
        seed=42,
        outdir="samples/classification_in",
        fair_voting=True,
        alpha=0.6,
        sensitive_feature=A_train, 
    )

ImportError: cannot import name 'PipegenieClassifier' from 'pipegenie.fairness' (C:\fairlearn\PipeGenie-main\pipegenie\fairness\__init__.py)

## Train the Model

In [5]:
model.fit(X_train,y_train)

                                       fitness                                            size                                        fitness_elite                                      size_elite              
                 ----------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max           avg           std           min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.50973       0.71908       0.65785       0.076968      1      5      2.64          1.3381        0.50973       0.71908       0.65785       0.076968      1      4      2             1.2649    
1      50        0.50973       0.71908       0.66597       0.062406      1      5      2.74          1.3388        0.67798       0.71908       0.69895       0.0

## Predict on the Test Set

In [6]:
y_pred = model.predict(X_test)

## Performance Metrics

In [ ]:
# =====================================================
# FAIRNESS REPORT
# =====================================================

from pipegenie.fairness import fairness_report

report = fairness_report(
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test,
    metric=balanced_accuracy_score
)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])


NameError: name 'y_test' is not defined

## Fairness Metrics

In [ ]:
# =====================================================
# STANDARD METRICS
# =====================================================
from pipegenie.metrics import balanced_accuracy_score
from fairlearn.metrics import (equalized_odds_difference,demographic_parity_difference)

print("\n==============================")
print("BALANCED ACCURACY")
print("==============================")

print(
    balanced_accuracy_score(
        y_test,
        y_pred
    )
)

print(
    "Demographic Parity Difference:",
    demographic_parity_difference(
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=A_test,
    ),
)

print(
    "Equalized Odds Difference:",
    equalized_odds_difference(
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=A_test,
    ),
)

## How Fair Voting Works

The standard VotingClassifier assigns the same importance to every estimator in the ensemble.

FairVotingClassifier modifies this behaviour by assigning a fairness-aware weight to every estimator.

The procedure is:

1. Train every estimator in the ensemble.
2. Evaluate each estimator on a validation set.
3. Compute:
   - Predictive performance (accuracy)
   - Fairness (Demographic Parity Difference)
4. Compute a fairness-aware weight

weight = α × accuracy + (1 − α) × fairness_score

where

fairness_score = 1 − |Demographic Parity Difference|

5. Normalize the weights.
6. Perform a weighted vote using these fairness-aware weights.

As a result, estimators with a better balance between predictive performance and fairness have a larger influence on the final prediction.

## Conclusion

This tutorial demonstrated the fairness-aware voting strategy integrated into PipeGenie.

Instead of relying on a standard majority vote, the ensemble combines predictions using weights computed from both predictive performance and fairness, allowing users to control the trade-off between these objectives through the `alpha` parameter.